In [ ]:
# study2_run_all.ipynb -- Study 2, all stages in one notebook (Era 2 daily
# correlation -> Era 3 EDA -> feature table -> violation baseline -> ramp-shock
# baseline), for anyone who just wants to open ONE file and run it end to end
# instead of hunting through 5 separate notebooks.
#
# The 5 original notebooks (00_era2_daily_correlation.ipynb, 01_eda.ipynb,
# 02_features.ipynb, 03_violation_baseline.ipynb, 04_ramp_shock_baseline.ipynb)
# still exist and are still the source of truth for each individual stage --
# this is a convenience wrapper, not a replacement.
#
# CREDENTIALS: this cell reads your Kaggle username/key from Colab's Secrets
# manager (the key icon in the left sidebar), NOT from a hardcoded string in this
# cell. Add two secrets named KAGGLE_USERNAME and KAGGLE_KEY there (toggle
# "Notebook access" on for each) before running this cell. Never type your actual
# key into a notebook cell you plan to commit -- this repo is public, and
# anything typed into a committed cell is visible to everyone and stays in git
# history even after you remove it later.
#
# NOTE: clones the FULL repo (not just ML/Study2) -- ML/Study2/features.py loads
# ML/Study1/features.py directly off disk, so Study 1's code needs to be present
# too even if you only care about Study 2's results.

!git clone https://github.com/HalcyonVector/Grid-Sentinel.git
%cd Grid-Sentinel/ML/Study2
!pip install lightgbm -q

import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip


In [ ]:
# 00_era2_daily_correlation.ipynb -- Era 2 (2023-Oct 2024) daily corridor/cross-border vs frequency-stress correlation
# Re-download if runtime reset:
# !kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip

import numpy as np
import pandas as pd

df = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

# Era 2 window: IR-line corridor cols are populated from 2023-01-01, cross-border cols
# from 2023-07-06 (both verified in Phase 3's audit) -- Era 2 ends where Era 3's live
# SCADA data begins (study2_scada starts 2024-11-04), so this window uses everything
# up to end-Oct 2024, the last full month before that handoff.
era2 = df[(df["date"] >= "2023-01-01") & (df["date"] <= "2024-10-31")].copy()

IR_COLS = [c for c in df.columns if c.startswith("ir_") and c.endswith("_net_mu")]
XB_COLS = [c for c in df.columns if c.startswith("xb_net_")]

# Stress proxy: % of the day spent outside the 49.9-50.05 Hz normal band (below 49.9 +
# above 50.05), from the existing freq_pct_* columns. Higher = more frequency instability.
era2["stress_pct"] = era2["freq_pct_below_499"] + era2["freq_pct_above_5005"]

print("Era 2 window:", era2.shape, era2["date"].min().date(), "->", era2["date"].max().date())

# --- IR-line (inter-regional) corridor congestion vs same-day stress ---
ir_sub = era2.dropna(subset=IR_COLS + ["stress_pct"])
print(f"\nIR-line rows usable: {len(ir_sub)} of {len(era2)}")
ir_abs_sum = ir_sub[IR_COLS].abs().sum(axis=1)
print("corr(sum|ir_net|, same-day stress_pct):", round(np.corrcoef(ir_abs_sum, ir_sub["stress_pct"])[0, 1], 3))
for c in IR_COLS:
    print(" ", c, round(np.corrcoef(ir_sub[c].abs(), ir_sub["stress_pct"])[0, 1], 3))

# --- Lagged version: does today's corridor flow predict TOMORROW's stress? ---
ir_lag = ir_sub.copy()
ir_lag["ir_abs_sum"] = ir_abs_sum
ir_lag = ir_lag.sort_values("date")
ir_lag["stress_pct_next"] = ir_lag["stress_pct"].shift(-1)
ir_lag_valid = ir_lag.dropna(subset=["stress_pct_next"])
print("\ncorr(ir_abs_sum today, stress_pct tomorrow):",
      round(np.corrcoef(ir_lag_valid["ir_abs_sum"], ir_lag_valid["stress_pct_next"])[0, 1], 3))

# --- Cross-border exchange vs same-day stress (from 2023-07-06 onward only) ---
xb_win = era2[era2["date"] >= "2023-07-06"].copy()
xb_sub = xb_win.dropna(subset=XB_COLS + ["stress_pct"])
print(f"\nCross-border rows usable: {len(xb_sub)} of {len(xb_win)}")
print("corr(sum|xb_net|, same-day stress_pct):",
      round(np.corrcoef(xb_sub[XB_COLS].abs().sum(axis=1), xb_sub["stress_pct"])[0, 1], 3))
for c in XB_COLS:
    v = xb_sub[c].abs()
    corr = np.corrcoef(v, xb_sub["stress_pct"])[0, 1] if v.std() > 0 else float("nan")
    print(" ", c, round(corr, 3) if corr == corr else "nan (zero variance -- no exchange recorded in this window)")

# --- Findings (verified 2026-07-11) ---
# IR-line corridor congestion has a moderate NEGATIVE correlation with same-day frequency
# stress (~-0.40 for the summed absolute net flow across all 7 corridors), strongest for
# WR<->NR (-0.42) and NER<->NR (-0.37). The lagged (today's flow vs tomorrow's stress)
# version is similar or slightly stronger (~-0.43). Negative direction makes physical
# sense: corridors move power specifically to relieve regional imbalance, so higher
# corridor utilization coincides with LOWER frequency instability, not higher -- i.e.
# corridors are evidence of the grid actively correcting stress, not causing it.
#
# Cross-border exchange shows close to no linear correlation with stress (~-0.01 to
# -0.17 depending on country; Myanmar's column is constant zero in this window --
# essentially no exchange recorded, consistent with it not being meaningfully
# grid-connected in the data). Bangladesh is the only country with even a mild signal
# (-0.17). This is a real, negative finding, not a null result to paper over: at DAILY
# resolution, cross-border exchange volume alone does not explain frequency stress in
# this window -- if it matters, it likely shows up at finer (SCADA) resolution or via a
# different framing (e.g. exchange volatility rather than level), which Era 3's live
# model (Phase 4 Era 3) is positioned to test.
#
# Feature-design takeaway for Era 3: IR-line net-flow magnitude (especially WR-NR and
# NER-NR) is the more promising corridor signal to carry into the SCADA-resolution
# classifier; cross-border columns are included there too but expectations should be
# calibrated low based on this daily-resolution pre-check.


In [ ]:
# 01_eda.ipynb -- Era 3 (Nov 2024-present) SCADA EDA: violation + ramp-shock rate by
# hour, season, gen-mix, and corridor stress
# Re-download if runtime reset:
# !kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip
# !pip install lightgbm -q

import pandas as pd
import matplotlib.pyplot as plt

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)

df = f.drop_bad_days(scada)  # drops 2024-11-20, 2025-04-01 (1 slot), 2025-10-02 (63 slots)
df = f.add_datetime(df)
df = f.add_violation_label(df)
df = f.add_ramp_label(df)
df = f.add_time_features(df)

print("rows after dropping bad days:", len(df), "of", len(scada))
print("overall violation rate:", round(df["violation"].mean(), 4))
print("overall ramp rate:", round(df["ramp"].mean(), 4))

# --- Rate by hour of day ---
by_hour = df.groupby("hour")[["violation", "ramp"]].mean()
import matplotlib as mpl
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

norm_v = mpl.colors.Normalize(vmin=by_hour["violation"].min(), vmax=by_hour["violation"].max())
axes[0].plot(by_hour.index, by_hour["violation"], color="#4a3aa7", alpha=0.25, linewidth=1)
axes[0].scatter(by_hour.index, by_hour["violation"], c=by_hour["violation"], cmap="RdPu", s=40, zorder=3)
axes[0].set_title("Frequency-violation rate by hour")

norm_r = mpl.colors.Normalize(vmin=by_hour["ramp"].min(), vmax=by_hour["ramp"].max())
axes[1].plot(by_hour.index, by_hour["ramp"], color="#4a3aa7", alpha=0.25, linewidth=1)
axes[1].scatter(by_hour.index, by_hour["ramp"], c=by_hour["ramp"], cmap="cool", s=40, zorder=3)
axes[1].set_title("Ramp-shock rate by hour")
plt.tight_layout()
plt.savefig("era3_rate_by_hour.png")
print(by_hour.round(4))

# --- Rate by month (seasonality) ---
by_month = df.groupby("month")[["violation", "ramp"]].mean()
print("\n", by_month.round(4))

# --- Solar-hour vs non-solar-hour ---
print("\nviolation rate, solar vs non-solar hour:")
print(df.groupby("is_solar_hr")["violation"].mean().round(4))
print("ramp rate, solar vs non-solar hour:")
print(df.groupby("is_solar_hr")["ramp"].mean().round(4))

# --- Weekday vs weekend ---
print("\nviolation rate, weekday(0) vs weekend(1):")
print(df.groupby("is_weekend")["violation"].mean().round(4))

# --- Generation mix: RES share vs event rate ---
df["res_bin"] = pd.qcut(df["share_res_pct"], 5, duplicates="drop")
print("\nviolation/ramp rate by RES-share quintile (low -> high):")
print(df.groupby("res_bin", observed=True)[["violation", "ramp"]].mean().round(4))

# --- Corridor congestion: sum|ir_net| vs event rate ---
ir_abs_sum = df[f.CORRIDOR_COLS[:7]].abs().sum(axis=1)  # the 7 ir_* corridor cols
df["ir_bin"] = pd.qcut(ir_abs_sum, 5, duplicates="drop")
print("\nviolation/ramp rate by corridor-flow quintile (low -> high):")
print(df.groupby("ir_bin", observed=True)[["violation", "ramp"]].mean().round(4))

# --- Findings (verified 2026-07-11 against live study2_scada.csv, 56,892 usable slots
#     after dropping the 3 corrupted-file days) ---
#
# Overall base rates: violation 0.89%, ramp-shock 6.1% -- both rare-event but workable
# class balances (matches the 0.88% violation rate measured earlier in the roadmap).
#
# STRONG time-of-day pattern, and it lines up with solar ramp physics, not noise:
#   - Violations cluster 07:00-14:00, peaking at 13:00 (4.0%) and 08:00-09:00 (~3%) --
#     the mid-morning-to-early-afternoon window where solar output is both large and
#     fast-changing (cloud transients, ramp-up/plateau).
#   - Ramp-shocks cluster in two bands: 05:00-09:00 (sunrise ramp-up, peaking 36% at
#     06:00) and 17:00-20:00 (sunset ramp-down, up to 8.6%) -- the two times of day solar
#     generation changes fastest. This is a clean, physically-explainable signal, not an
#     artifact -- and it's the single strongest predictor a lead-time model should exploit
#     (confirmed in 03/04's feature importance: "hour" is the top feature for ramp-shock).
#   - Solar-hour violation rate (1.53%) is ~6x the non-solar-hour rate (0.26%).
#
# RES share vs violation rate is MONOTONIC and increasing: 0.33% in the lowest RES-share
# quintile up to 1.92% in the highest -- direct SCADA-resolution evidence for the
# project's central "rising RES share stresses the grid" thesis (Era 1 found a similar,
# weaker signal at daily/monthly resolution; this is the live, granular version of it).
#
# RES share vs ramp-shock rate goes the OTHER way in this simple quintile binning (8.3%
# in lowest quintile down to 2.8% in highest) -- flagged as a genuine, unresolved,
# counter-intuitive finding, not smoothed over: RES share is itself strongly seasonal
# (higher in summer), and month is independently a strong driver of ramp rate (Jan/Dec
# ~13-15%, Jul ~0.7%), so this crude binning is very likely confounded by season rather
# than showing a true RES effect. A cleaner month-controlled analysis is future work, not
# resolved here.
#
# Corridor-flow (ir_abs_sum) quintiles show no clean monotonic relationship with either
# event rate in this simple binning -- consistent with Era 2's daily-resolution finding
# that corridor flow's relationship to stress is real but not simply "more flow = more
# events" (it's corridors correcting stress, not causing it). The lead-time classifiers
# in 03/04 use the raw per-corridor columns rather than this aggregate, which is
# expected to capture more of that relationship than a single summed quintile can.


In [ ]:
# --- Appendix: hour x day-of-week heatmap, and resolving the RES-share/ramp finding
#     (added 2026-07-11) ---
# The single-axis breakdowns above (hour alone, month alone) can hide interaction
# effects. This cross-tabs hour against day-of-week, and separately resolves the
# "ramp rate falls with RES-share quintile" finding flagged above as likely
# season-confounded -- by actually controlling for season instead of leaving it open.

import numpy as np
import matplotlib.pyplot as plt

print("=== violation rate: hour (columns) x day-of-week (rows, 0=Mon..6=Sun) ===")
viol_heatmap = df.pivot_table(index="dow", columns="hour", values="violation", aggfunc="mean")
print(viol_heatmap.round(3).to_string())

print("\n=== ramp rate: hour (columns) x day-of-week (rows, 0=Mon..6=Sun) ===")
ramp_heatmap = df.pivot_table(index="dow", columns="hour", values="ramp", aggfunc="mean")
print(ramp_heatmap.round(3).to_string())

print("\nmarginal violation rate by day-of-week:", df.groupby("dow")["violation"].mean().round(4).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
im0 = axes[0].imshow(viol_heatmap.values, cmap="RdPu", aspect="auto")
axes[0].set_xticks(range(len(viol_heatmap.columns)), viol_heatmap.columns)
axes[0].set_yticks(range(len(viol_heatmap.index)), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[0].set_xlabel("hour")
axes[0].set_title("Violation rate: hour x day-of-week")
fig.colorbar(im0, ax=axes[0], label="violation rate")

im1 = axes[1].imshow(ramp_heatmap.values, cmap="cool", aspect="auto")
axes[1].set_xticks(range(len(ramp_heatmap.columns)), ramp_heatmap.columns)
axes[1].set_yticks(range(len(ramp_heatmap.index)), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[1].set_xlabel("hour")
axes[1].set_title("Ramp-shock rate: hour x day-of-week")
fig.colorbar(im1, ax=axes[1], label="ramp rate")

plt.tight_layout()
plt.savefig("era3_heatmap_hour_dow.png")
plt.show()
print("marginal ramp rate by day-of-week:", df.groupby("dow")["ramp"].mean().round(4).to_dict())

# --- Season-controlled RES-share vs ramp/violation rate ---
# Instead of a global RES-share quintile (confounded by season -- RES share and ramp
# rate are both independently seasonal), rank RES share WITHIN each month first, then
# quintile that rank. This isolates "is this day unusually high/low RES for its own
# season" from "which month is it."
df["res_rank_in_month"] = df.groupby("month")["share_res_pct"].rank(pct=True)
df["res_bin_within_month"] = pd.cut(df["res_rank_in_month"], 5, labels=False)
within_month = df.groupby("res_bin_within_month", observed=True)[["violation", "ramp"]].mean()
print("\nviolation/ramp rate by RES-share quintile, WITHIN month (season-controlled):")
print(within_month.round(4))

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(5)
width = 0.35
pooled_ramp = df.groupby("res_bin", observed=True)["ramp"].mean().values
ax.bar(x - width / 2, pooled_ramp, width, label="pooled (uncontrolled)", color="#9085e9")
ax.bar(x + width / 2, within_month["ramp"].values, width, label="within-month (season-controlled)", color="#4a3aa7")
ax.set_xticks(x, ["Q1 (low)", "Q2", "Q3", "Q4", "Q5 (high)"])
ax.set_xlabel("RES-share quintile")
ax.set_ylabel("ramp-shock rate")
ax.set_title("RES-share vs ramp-shock rate: pooled sign reverses once season is controlled for")
ax.legend()
plt.tight_layout()
plt.savefig("era3_res_share_ramp_reversal.png")
plt.show()

# Partial correlation: residualize both RES share and the outcome against month dummies
# (linear regression on month, keep the residuals), then correlate the residuals. This
# is the same idea as the within-month quintile above but as a single number.
month_dummies = pd.get_dummies(df["month"], prefix="m", drop_first=True).astype(float)
X = np.column_stack([np.ones(len(df)), month_dummies.values])

def residualize(y):
    y = y.values.astype(float)
    mask = ~np.isnan(y)
    beta, *_ = np.linalg.lstsq(X[mask], y[mask], rcond=None)
    resid = np.full(len(y), np.nan)
    resid[mask] = y[mask] - X[mask] @ beta
    return resid

res_share_resid = residualize(df["share_res_pct"])
ramp_resid = residualize(df["ramp"])
viol_resid = residualize(df["violation"])
m1 = ~np.isnan(res_share_resid) & ~np.isnan(ramp_resid)
m2 = ~np.isnan(res_share_resid) & ~np.isnan(viol_resid)
print("\npooled corr(share_res_pct, ramp):", round(df[["share_res_pct", "ramp"]].corr().iloc[0, 1], 4))
print("partial corr(share_res_pct, ramp | month):", round(np.corrcoef(res_share_resid[m1], ramp_resid[m1])[0, 1], 4))
print("partial corr(share_res_pct, violation | month):", round(np.corrcoef(res_share_resid[m2], viol_resid[m2])[0, 1], 4))

# --- Findings (verified 2026-07-11) ---
# Heatmap: violations concentrate hardest on Sunday (dow=6) -- the single worst slot is
# Sunday 13:00 at 7.0%, and Sunday's marginal violation rate (1.38%) is the highest of
# any day, well above the weekday range (0.62-0.99%). Ramp-shocks show the opposite
# day-of-week pattern: Sunday's marginal ramp rate (4.40%) is the LOWEST of the week,
# while every weekday sits around 6.4-6.7%. That's a genuinely interesting decoupling,
# not noise: Sunday has fewer large demand swings (lower industrial/commercial load,
# smoother curve) but MORE frequency violations -- consistent with the grid running a
# thinner online generation/reserve margin on low-demand days, making frequency more
# sensitive to whatever smaller disturbances do occur. Worth carrying into feature
# engineering or the paper's discussion, not just filed as a curiosity.
#
# RES-share/ramp confounding, RESOLVED: the earlier pooled finding (ramp rate falls
# from 8.3% to 2.8% across RES-share quintiles) was indeed season-confounded, as
# suspected -- and controlling for it doesn't just weaken the effect, it REVERSES it.
# Within-month (season-controlled) RES-share quintiles show ramp rate RISING from 5.3%
# to 7.0%, and the partial correlation (residualized on month) flips from -0.075
# (pooled) to +0.026 (season-controlled) -- small in magnitude, but the correct sign is
# now consistent with the violation-rate finding (partial corr +0.051) and with the
# project's central "rising RES share stresses the grid" thesis. The pooled/uncontrolled
# number was actively misleading, not just imprecise -- this is a case where the
# season-controlled analysis was necessary to get the right qualitative answer, not just
# a more precise one.


In [ ]:
# 02_features.ipynb -- Study 2 feature table: labels, lag/rolling slot features,
# corridor/cross-border broadcast, Study 1 residual signal, class balance
# !pip install lightgbm -q

import pandas as pd

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)

# build_feature_table() with no study1_residual argument computes it itself, by
# backtesting Study 1's baseline (train <=2022, val 2023, predict 2024+) --
# see features.py's build_study1_residual_signal() docstring for why this is a
# backtest approximation rather than a reproduction of predict.py's live daily output.
feat_df = f.build_feature_table(scada)

print("feature table shape:", feat_df.shape)
print("columns:", feat_df.columns.tolist())

print("\nviolation_lead: ", feat_df["violation_lead"].notna().sum(), "labeled rows, rate =",
      round(feat_df["violation_lead"].mean(), 4))
print("ramp_lead:      ", feat_df["ramp_lead"].notna().sum(), "labeled rows, rate =",
      round(feat_df["ramp_lead"].mean(), 4))

print("\nrows with unresolvable lead label (dropped before training):")
print(" violation_lead NaN:", feat_df["violation_lead"].isna().sum())
print(" ramp_lead NaN:", feat_df["ramp_lead"].isna().sum())

print("\nnull rate per feature column:")
print(feat_df[f.FEATURE_COLS].isna().mean().round(4).sort_values(ascending=False))

# --- Notes ---
# violation_lead and ramp_lead are both "did the event happen in any of the next 1-4
# slots (15-60 min)" -- OR'd over the lookahead window, so their positive rate is
# necessarily higher than the raw per-slot event rate from 01_eda.ipynb (0.89% ->
# ~2.4% for violation, 6.1% -> ~15% for ramp, since the OR captures any of 4 chances).
#
# study1_residual_mw nulls are concentrated at the very start (before Study 1's 2024
# backtest window begins) and the very end (today's row, whose "tomorrow" hasn't
# happened yet in study1_daily.csv) -- structural, not a data quality bug.
#
# LightGBM handles NaN features natively (treated as a learnable split direction), so
# no imputation is applied here -- imputing would risk inventing signal that isn't
# really there, especially study1_residual_mw, whose nulls are
# meaningfully informative (e.g. "no residual yet" vs "residual was exactly 0").
#
# share_res_pct and the 11 corridor/cross-border columns are computed here (they're
# still part of feat_df) but deliberately excluded from FEATURE_COLS -- see
# features.py's DAILY_BROADCAST_COLS comment: they are whole-day aggregates broadcast
# identically to all 96 slots of a day, and removing them from the classifiers'
# feature set improved both targets rather than hurting them (found 2026-07-11).


In [ ]:
# 03_violation_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# frequency-violation target
# !pip install lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

import features as f

TARGET = "violation_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()  # drop rows whose lead window is unresolvable

# --- Time-aware split ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# NOTE (2026-07-11): deliberately NOT using scale_pos_weight -- see
# features.py's scale_pos_weight() docstring for the full story.
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
print(f"\nbest_iteration_: {model.best_iteration_}  (sanity check -- should NOT be 1)")

proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"PR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

precision, recall, thresh = precision_recall_curve(y_test, proba_test)
f1s = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.nanargmax(f1s[:-1])
print(f"Best-F1 operating point: F1={f1s[best_idx]:.4f} at threshold={thresh[best_idx]:.4f} "
      f"(precision={precision[best_idx]:.4f}, recall={recall[best_idx]:.4f})")

idx95 = np.where(precision[:-1] >= 0.95)[0]
recall_at_95p = recall[idx95].max() if len(idx95) else 0.0
print(f"Recall at >=95% precision: {recall_at_95p:.4f}")

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

plt.figure(figsize=(8, 6))
top_imp = importance.head(15)
norm = mpl.colors.Normalize(vmin=top_imp.min(), vmax=top_imp.max())
colors = mpl.colormaps["RdPu"](norm(top_imp.values))
plt.barh(top_imp.index[::-1], top_imp.values[::-1], color=colors[::-1])
plt.xlabel("Split count")
plt.title("Feature importance -- frequency-violation lead-time classifier")
plt.tight_layout()
plt.show()

# --- Results (verified 2026-07-11, LightGBM 4.6.0; fourth version of this notebook's
#     numbers -- see full history below) ---
# best_iteration_: 65
# PR-AUC 0.1567 vs a random/base-rate baseline of 0.0305 -- 5.13x lift over chance.
# Best-F1 operating point: F1=0.2035, precision=17.7%, recall=24.0%.
#
# Full history of this notebook's numbers, in order:
#  1. PR-AUC 0.0614 -- scale_pos_weight silently limiting training to 1 boosting round.
#  2. PR-AUC 0.0937 -- fixed scale_pos_weight + added solar_delta_mw/solar_roll8_std
#     (after the two-stage ramp->violation hypothesis was tested and found false).
#  3. PR-AUC 0.1186 -- removed share_res_pct + 11 corridor/cross-border columns (whole-
#     day aggregates broadcast to every slot; tested as a leakage concern, found to be
#     pure noise instead -- removing them helped, not hurt).
#  4. THIS version, PR-AUC 0.1567 -- added freq_hz_delta and wind_delta_mw, the two
#     next-highest correlations with violation_lead from the same diagnostic scan that
#     originally found solar_delta_mw (freq_hz's own one-step delta: 0.0978, actually
#     the single highest correlation found in that whole scan; wind_delta_mw: 0.0361).
#     Both were identified back when solar_delta_mw was added but not acted on until
#     asked to add them explicitly. Real, verified gain: +32% PR-AUC over the previous
#     version, and both new features have genuine, nonzero importance in the trained
#     model (not just harmless noise that happened not to hurt).


In [ ]:
# --- Appendix: shorter lead-window experiment (2026-07-11, FOURTH pass -- re-verified
#     after adding freq_hz_delta/wind_delta_mw, see the main cell above) ---
# Does shrinking the lookahead window from 1-4 slots improve the violation classifier?
# features.py's add_violation_label() and build_feature_table() both take a lead_slots
# override for exactly this test.

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import average_precision_score, precision_recall_curve

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
resid = f.build_study1_residual_signal()  # computed once, reused across all window sizes


def run(lead_slots):
    feat = f.build_feature_table(scada, study1_residual=resid, violation_lead_slots=lead_slots)
    df = feat.dropna(subset=["violation_lead"]).copy()
    train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
    val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
    test = df[df["date"] >= "2026-01-01"]

    X_train, y_train = train[f.FEATURE_COLS], train["violation_lead"]
    X_val, y_val = val[f.FEATURE_COLS], val["violation_lead"]
    X_test, y_test = test[f.FEATURE_COLS], test["violation_lead"]

    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="average_precision",
              callbacks=[lgb.early_stopping(50, verbose=False)])

    proba = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, proba)
    base_rate = y_test.mean()

    precision, recall, thresh = precision_recall_curve(y_test, proba)
    f1s = 2 * precision * recall / (precision + recall + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])

    print(f"lead_slots={lead_slots}: best_iter={model.best_iteration_}  base_rate={base_rate:.4f}  "
          f"PR-AUC={pr_auc:.4f} (lift={pr_auc / base_rate:.2f}x)  best-F1={f1s[best_idx]:.4f} "
          f"(P={precision[best_idx]:.4f} R={recall[best_idx]:.4f})")


for k in [4, 3, 2, 1]:
    run(k)

# --- Findings (re-verified 2026-07-11, FOURTH pass -- supersedes all three earlier
#     versions of this appendix) ---
# lead_slots=4 (shipped, 15-60 min): PR-AUC=0.1567 (5.13x lift)  best-F1=0.2035 (P=17.7% R=24.0%)
# lead_slots=3 (15-45 min):          PR-AUC=0.1612 (6.55x lift)  best-F1=0.2097 (P=21.0% R=21.0%)
# lead_slots=2 (15-30 min):          PR-AUC=0.2106 (11.62x lift) best-F1=0.2750 (P=22.7% R=34.9%)
# lead_slots=1 (15 min only):        PR-AUC=0.2185 (19.71x lift) best-F1=0.3212 (P=28.1% R=37.5%)
#
# Notably different from the previous three passes: this is the first time the pattern
# is CLEANLY MONOTONIC -- PR-AUC, best-F1, precision, AND recall all improve together as
# the window shrinks from 4 to 1 slot. Previous passes showed unstable, sometimes
# contradictory rankings (2 slots best on one pass, 1 slot best on another, no clear
# winner on the first). With better features (this pass added freq_hz_delta and
# wind_delta_mw), the shorter-window advantage is no longer just a precision/recall
# trade-off -- 1 slot now dominates on every single metric, including recall, which
# earlier passes showed getting WORSE as the window shrank. That reversal is itself
# informative: a real, learnable, near-term signal exists that these two new features
# capture much better than the previous feature set could, and it's concentrated in the
# very next slot rather than spread evenly across the 4-slot window.
#
# Still not changing the shipped default without an explicit product decision -- a
# 15-minute-only warning is a meaningfully different product than a 15-60-minute one,
# and this is the fourth different ranking in four passes, so one more round of
# stability (e.g. does this monotonic pattern hold under cross-validation, not just one
# time-aware split) would be worth having before treating it as settled. But this is the
# strongest evidence yet that a shorter window is worth shipping.


In [ ]:
# 04_ramp_shock_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# ramp-shock target (same feature pipeline as 03_violation_baseline.ipynb)
# !pip install lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

import features as f

TARGET = "ramp_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()

# --- Time-aware split (same as 03_violation_baseline.ipynb) ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# NOTE (2026-07-11): NOT using scale_pos_weight -- see features.py's
# scale_pos_weight() docstring and 03_violation_baseline.ipynb for the full story.
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
print(f"\nbest_iteration_: {model.best_iteration_}")

proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"PR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

precision, recall, thresh = precision_recall_curve(y_test, proba_test)
f1s = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.nanargmax(f1s[:-1])
print(f"Best-F1 operating point: F1={f1s[best_idx]:.4f} at threshold={thresh[best_idx]:.4f} "
      f"(precision={precision[best_idx]:.4f}, recall={recall[best_idx]:.4f})")

idx95 = np.where(precision[:-1] >= 0.95)[0]
recall_at_95p = recall[idx95].max() if len(idx95) else 0.0
print(f"Recall at >=95% precision: {recall_at_95p:.4f}")

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

plt.figure(figsize=(8, 6))
top_imp = importance.head(15)
norm = mpl.colors.Normalize(vmin=top_imp.min(), vmax=top_imp.max())
colors = mpl.colormaps["cool"](norm(top_imp.values))
plt.barh(top_imp.index[::-1], top_imp.values[::-1], color=colors[::-1])
plt.xlabel("Split count")
plt.title("Feature importance -- ramp-shock lead-time classifier")
plt.tight_layout()
plt.show()

# --- Results (verified 2026-07-11, LightGBM 4.6.0; fourth version of this notebook's
#     numbers -- see full history below) ---
# PR-AUC 0.7486 vs a random/base-rate baseline of 0.1793 -- 4.18x lift over chance.
# F1@0.5 = 0.5619. Best-F1 operating point (threshold ~0.21): F1=0.6803, precision=63.3%,
# recall=73.5%. Recall at >=95% precision = 20.4%.
#
# Full history of this notebook's numbers: (1) PR-AUC 0.7140 with scale_pos_weight;
# (2) 0.7248 after removing it + adding solar_delta_mw/solar_roll8_std; (3) 0.7446 after
# removing share_res_pct + 11 corridor/cross-border columns (whole-day aggregates
# broadcast to every slot -- tested as a leakage concern, found to be pure noise
# instead); (4) THIS version, 0.7486, after adding freq_hz_delta and wind_delta_mw (the
# two next-highest correlations with violation_lead from the diagnostic scan that
# originally found solar_delta_mw). Smaller gain here than for violation_lead (see
# 03_violation_baseline.ipynb, +32% PR-AUC there) -- wind_delta_mw contributes real,
# nonzero importance (rank 12) but freq_hz_delta barely registers for this target,
# unlike for violation. Makes sense: this target is about DEMAND swings, and freq_hz's
# own volatility is a more direct signal for frequency-adjacent problems (violation)
# than for demand-driven ones (ramp-shock).
#
# Top features: hour, demand_delta_mw, month, demand_met_mw_lag3, solar_delta_mw,
# solar_roll8_std -- solar and demand-trajectory features dominate, consistent with
# 01_eda.ipynb's sunrise/sunset clustering finding. Corridor columns no longer appear at
# all (removed from FEATURE_COLS) -- Era 2's daily-resolution corridor-flow finding
# remains a separate, valid, unaffected result, it just isn't what drives this live
# classifier.
#
# This target remains markedly easier to predict with lead time than frequency
# violation (PR-AUC 0.1567 even after the same four rounds of fixes) -- plausibly
# because a ramp-shock is a direct, mechanical property of the demand/generation
# trajectory itself, while a frequency violation is a downstream consequence that
# depends on how well AGC/reserves absorb a given ramp, adding a layer of noise the raw
# features here don't fully capture.
